# House Prices — Advanced Regression Techniques

**Goal:** Predict the sales price for each house based on 79 explanatory variables.

**Metric:** RMSE (Root Mean Squared Error on log-transformed prices).

**Kaggle link:** https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

## 1. Setup & Imports

In [ ]:
# Standard library
import warnings
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt

# Data manipulation
import numpy as np
import pandas as pd
import seaborn as sns

# Settings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

%matplotlib inline

## 2. Configuration

In [ ]:
# Project paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../outputs/models")
SUBMISSIONS_DIR = Path("../outputs/submissions")

# Competition settings
COMPETITION_NAME = "house-prices-advanced-regression-techniques"
TARGET_COL = "SalePrice"
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 3. Load Data

In [ ]:
train_df = pd.read_csv(DATA_RAW / "train.csv")
test_df = pd.read_csv(DATA_RAW / "test.csv")

print(
    f"Train shape: {train_df.shape} — {train_df.shape[0]} houses, {train_df.shape[1]} columns (79 features + Id + SalePrice)"
)
print(
    f"Test shape:  {test_df.shape} — {test_df.shape[0]} houses, {test_df.shape[1]} columns (79 features + Id, no SalePrice)"
)

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train_df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"\nNumeric features: {len(numeric_cols) - 2} (excluding Id and SalePrice)")
print(f"Categorical features: {len(cat_cols)}")

## 4. Exploratory Data Analysis (EDA)

**Rules:** No modeling, no cleaning — only observation. Document every finding.

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

### 4.1 Missing Values

In [ ]:
# Missing values analysis for train and test
def missing_summary(df, name=""):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    result = pd.DataFrame({"count": missing, "percentage": missing_pct})
    result = result[result["count"] > 0].sort_values("percentage", ascending=False)
    print(f"=== {name} — {len(result)} columns with missing values ===")
    return result


train_missing = missing_summary(train_df, "Train")
train_missing

### 4.2 Target Distribution (SalePrice)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw distribution
train_df[TARGET_COL].hist(bins=50, ax=axes[0], edgecolor="black")
axes[0].set_title(f"{TARGET_COL} — Raw Distribution")
axes[0].set_xlabel("Price ($)")
axes[0].axvline(
    train_df[TARGET_COL].mean(), color="red", linestyle="--", label=f"Mean: ${train_df[TARGET_COL].mean():,.0f}"
)
axes[0].axvline(
    train_df[TARGET_COL].median(), color="green", linestyle="--", label=f"Median: ${train_df[TARGET_COL].median():,.0f}"
)
axes[0].legend()

# Log-transformed distribution (this is what Kaggle evaluates)
log_prices = np.log1p(train_df[TARGET_COL])
log_prices.hist(bins=50, ax=axes[1], edgecolor="black", color="orange")
axes[1].set_title(f"log(1 + {TARGET_COL}) — Closer to Normal")
axes[1].set_xlabel("Log Price")

# QQ plot of log-transformed prices
from scipy import stats

stats.probplot(log_prices, plot=axes[2])
axes[2].set_title("QQ Plot — log(1 + SalePrice)")

plt.tight_layout()
plt.show()

print(f"SalePrice range: ${train_df[TARGET_COL].min():,} — ${train_df[TARGET_COL].max():,}")
print(f"Mean: ${train_df[TARGET_COL].mean():,.0f} | Median: ${train_df[TARGET_COL].median():,.0f}")
print(f"Skewness (raw): {train_df[TARGET_COL].skew():.3f} → right-skewed, log transform helps")
print(f"Skewness (log): {log_prices.skew():.3f} → much closer to normal")

### 4.3 Correlations with SalePrice

In [ ]:
# Top 15 features most correlated with SalePrice
corr_with_target = (
    train_df.select_dtypes(include=[np.number]).corr()[TARGET_COL].drop([TARGET_COL, "Id"]).sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
top_15 = corr_with_target.head(15)
sns.barplot(x=top_15.values, y=top_15.index, palette="viridis", ax=ax)
ax.set_title("Top 15 Numeric Features — Correlation with SalePrice")
ax.set_xlabel("Pearson Correlation")
for i, v in enumerate(top_15.values):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
plt.tight_layout()
plt.show()

print("Key insight: OverallQual (0.79) and GrLivArea (0.71) are the strongest predictors.")
print("Garage, basement, and year features also matter significantly.")

In [ ]:
# Correlation heatmap — top 10 features + SalePrice
top_features = corr_with_target.head(10).index.tolist() + [TARGET_COL]
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    train_df[top_features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax,
    square=True,
    linewidths=0.5,
)
ax.set_title("Correlation Heatmap — Top 10 Features + SalePrice")
plt.tight_layout()
plt.show()

# Watch for multicollinearity
print("Multicollinearity to watch:")
print("  - GarageCars ↔ GarageArea (0.88) — keep one")
print("  - TotalBsmtSF ↔ 1stFlrSF (0.82) — related")
print("  - GrLivArea ↔ TotRmsAbvGrd (0.83) — related")
print("  - YearBuilt ↔ GarageYrBlt (0.83) — garage built same year as house")

### 4.4 Key Feature Relationships with SalePrice

In [ ]:
# Scatter plots — top 4 numeric features vs SalePrice
top_4 = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, col in enumerate(top_4):
    axes[i].scatter(train_df[col], train_df[TARGET_COL], alpha=0.3, s=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel(TARGET_COL if i == 0 else "")
    axes[i].set_title(f"{col} vs {TARGET_COL}")

plt.tight_layout()
plt.show()

# GrLivArea outliers: two houses >4000 sqft with low price — likely partial sales
outliers = train_df[(train_df["GrLivArea"] > 4000) & (train_df[TARGET_COL] < 300000)]
print(f"Potential outliers: {len(outliers)} houses with GrLivArea > 4000 sqft but SalePrice < $300k")
print(outliers[["Id", "GrLivArea", TARGET_COL, "SaleCondition"]])

### 4.5 Categorical Features

In [ ]:
# Categorical features — cardinality overview
cat_info = pd.DataFrame(
    {
        "nunique": train_df[cat_cols].nunique(),
        "missing_pct": (train_df[cat_cols].isnull().sum() / len(train_df) * 100).round(1),
    }
).sort_values("nunique", ascending=False)
print("Categorical features — cardinality and missing %:")
cat_info

In [ ]:
# Top categorical features — median SalePrice by category
key_cats = ["Neighborhood", "ExterQual", "KitchenQual", "BsmtQual", "GarageFinish"]
fig, axes = plt.subplots(1, len(key_cats), figsize=(24, 6))

for i, col in enumerate(key_cats):
    order = train_df.groupby(col)[TARGET_COL].median().sort_values(ascending=False).index
    sns.boxplot(data=train_df, x=col, y=TARGET_COL, order=order, ax=axes[i])
    axes[i].set_title(f"{TARGET_COL} by {col}")
    axes[i].tick_params(axis="x", rotation=45)
    if i > 0:
        axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

print("Key insight: Quality features (ExterQual, KitchenQual, BsmtQual) show clear price ordering.")
print("Neighborhood is the strongest categorical predictor — large price variation between neighborhoods.")

### 4.6 EDA Summary

**Dataset:** 1460 train / 1459 test houses, 79 features (36 numeric + 43 categorical).

**Target (SalePrice):**
- Range: $34,900 — $755,000 (median $163,000)
- Right-skewed (1.88) → log transform makes it near-normal (0.12)
- Kaggle evaluates RMSE on log-prices, so we should train on log(SalePrice)

**Missing values:**
- 19 columns have missing values in train
- PoolQC (99.5%), MiscFeature (96.3%), Alley (93.8%), Fence (80.8%): NA means "absent"
- MasVnrType (59.7%): NA likely means no masonry veneer
- Garage* (5.5%) and Bsmt* (2.5%): NA means no garage / no basement
- LotFrontage (17.7%): genuinely missing, impute by neighborhood median
- Electrical (1 row): impute with mode

**Top numeric predictors:** OverallQual (0.79), GrLivArea (0.71), GarageCars (0.64), GarageArea (0.62), TotalBsmtSF (0.61)

**Multicollinearity:** GarageCars↔GarageArea, TotalBsmtSF↔1stFlrSF, GrLivArea↔TotRmsAbvGrd — may need to drop one of each pair

**Top categorical predictors:** Neighborhood, ExterQual, KitchenQual, BsmtQual, GarageFinish

**Outliers:** 2 houses with GrLivArea > 4000 sqft but SalePrice < $300k — consider removing

**Next steps (Data Cleaning):**
1. Handle "absent" NAs (fill with "None" for categorical, 0 for numeric)
2. Impute LotFrontage by neighborhood median
3. Impute remaining missing values (mode for categorical, median for numeric)
4. Consider removing the 2 GrLivArea outliers

## 5. Data Cleaning

In [ ]:
# Remove 2 GrLivArea outliers identified in EDA (>4000 sqft, <$300k)
# WHY: These are likely partial sales that would distort the model
outlier_idx = train_df[(train_df["GrLivArea"] > 4000) & (train_df[TARGET_COL] < 300000)].index
print(f"Removing {len(outlier_idx)} outliers (Ids: {train_df.loc[outlier_idx, 'Id'].tolist()})")
train_df = train_df.drop(outlier_idx).reset_index(drop=True)
print(f"Train shape after outlier removal: {train_df.shape}")

### 5.1 Handle "Feature Absent" NAs

Per `data_description.txt`, NA in many columns means the feature doesn't exist (no pool, no garage, no basement, etc.). These are not truly missing — fill with `"None"` (categorical) or `0` (numeric).

In [ ]:
# Columns where NA means "feature absent" — fill with "None" or 0
# WHY: data_description.txt documents NA as a valid category (e.g., "NA = No Pool")

absent_cat_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType",
]
absent_num_cols = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
    "MasVnrArea",
]

for df in [train_df, test_df]:
    for col in absent_cat_cols:
        df[col] = df[col].fillna("None")
    for col in absent_num_cols:
        df[col] = df[col].fillna(0)

print("Filled 'feature absent' NAs:")
print(f"  Categorical ({len(absent_cat_cols)} cols): → 'None'")
print(f"  Numeric ({len(absent_num_cols)} cols): → 0")

### 5.2 Impute LotFrontage by Neighborhood Median

WHY: Lots in the same neighborhood tend to have similar frontage. Neighborhood median is a better imputation than global median.

In [ ]:
# Impute LotFrontage — fit on train, apply to both
# WHY: No data leakage — stats computed on train only
lot_frontage_by_neighborhood = train_df.groupby("Neighborhood")["LotFrontage"].median()

for df in [train_df, test_df]:
    for idx in df[df["LotFrontage"].isnull()].index:
        neighborhood = df.loc[idx, "Neighborhood"]
        median_val = lot_frontage_by_neighborhood.get(neighborhood, train_df["LotFrontage"].median())
        df.loc[idx, "LotFrontage"] = median_val

print("LotFrontage imputed by neighborhood median (fitted on train)")
print(f"  Train missing: {train_df['LotFrontage'].isnull().sum()}")
print(f"  Test missing:  {test_df['LotFrontage'].isnull().sum()}")

### 5.3 Impute Remaining Missing Values

Strategy: mode for categorical, median for numeric. All stats fitted on train only.

In [ ]:
# Impute remaining missing values — fit on train, apply to both
# WHY: Some columns have a few remaining NAs (Electrical in train, MSZoning/Functional/etc. in test)

# Compute fill values from train
cat_fill = {}
num_fill = {}

for col in train_df.columns:
    if col in ["Id", TARGET_COL]:
        continue
    if train_df[col].dtype.kind in ("f", "i", "u"):  # float, int, unsigned int
        num_fill[col] = train_df[col].median()
    else:
        cat_fill[col] = train_df[col].mode().iloc[0] if not train_df[col].mode().empty else "None"

# Apply to both datasets
for df in [train_df, test_df]:
    for col, val in cat_fill.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    for col, val in num_fill.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)

print("Remaining NAs imputed (mode for categorical, median for numeric)")
print(f"  Train total missing: {train_df.drop(columns=['Id', TARGET_COL]).isnull().sum().sum()}")
print(f"  Test total missing:  {test_df.drop(columns=['Id']).isnull().sum().sum()}")

### 5.4 Verification

In [ ]:
# Final verification — zero missing values in both datasets
train_missing_final = train_df.drop(columns=["Id", TARGET_COL]).isnull().sum().sum()
test_missing_final = test_df.drop(columns=["Id"]).isnull().sum().sum()

print(f"Train missing values: {train_missing_final}")
print(f"Test missing values:  {test_missing_final}")
assert train_missing_final == 0, "Train still has missing values!"
assert test_missing_final == 0, "Test still has missing values!"
print("\n✓ No missing values remain in train or test.")

print("\nFinal shapes:")
print(f"  Train: {train_df.shape}")
print(f"  Test:  {test_df.shape}")

## 6. Feature Engineering

In [ ]:
# TODO: Create new features
# Examples:
# train_df["new_feature"] = train_df["col_a"] / train_df["col_b"]
# train_df["is_weekend"] = train_df["day_of_week"].isin([5, 6]).astype(int)

# TODO: Encode categorical variables
# le = LabelEncoder()
# train_df["category_encoded"] = le.fit_transform(train_df["category"])

# TODO: Apply same transformations to test_df!

## 7. Modeling

In [ ]:
# Prepare features and target
# TODO: Select your feature columns
# feature_cols = ["col1", "col2", "col3"]
# X = train_df[feature_cols]
# y = train_df[TARGET_COL]

# Train/validation split
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
# )
# print(f"Train: {X_train.shape}, Validation: {X_val.shape}")

In [ ]:
# Baseline model
# TODO: Choose your baseline model

# For classification:
# model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
# model.fit(X_train, y_train)

# Cross-validation
# cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
# print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 8. Evaluation

In [ ]:
# Predictions on validation set
# y_pred = model.predict(X_val)

# For classification:
# print(f"Accuracy: {accuracy_score(y_val, y_pred):.4f}")
# print(f"\nClassification Report:")
# print(classification_report(y_val, y_pred))

# Confusion matrix
# fig, ax = plt.subplots(figsize=(8, 6))
# cm = confusion_matrix(y_val, y_pred)
# sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
# ax.set_title("Confusion Matrix")
# ax.set_ylabel("Actual")
# ax.set_xlabel("Predicted")
# plt.tight_layout()
# plt.show()

In [ ]:
# Feature importance
# TODO: Uncomment after training a tree-based model

# importance_df = pd.DataFrame({
#     "feature": feature_cols,
#     "importance": model.feature_importances_
# }).sort_values("importance", ascending=False)

# fig, ax = plt.subplots(figsize=(10, 6))
# sns.barplot(data=importance_df, x="importance", y="feature", ax=ax)
# ax.set_title("Feature Importance")
# plt.tight_layout()
# plt.show()

## 9. Submission

In [ ]:
# Generate predictions on test set
# TODO: Apply same preprocessing to test_df
# X_test = test_df[feature_cols]
# test_predictions = model.predict(X_test)

# Create submission file
# TODO: Adapt columns to match competition requirements
# submission = pd.DataFrame({
#     "Id": test_df["Id"],
#     TARGET_COL: test_predictions
# })

# submission.to_csv(SUBMISSIONS_DIR / "submission.csv", index=False)
# print(f"Submission saved to {SUBMISSIONS_DIR / 'submission.csv'}")
# print(f"Shape: {submission.shape}")
# submission.head()